# API and Web Scraping

This notebook covers how to work with APIs and web scraping using Python. We'll explore making API requests, parsing JSON data, and extracting information from web pages using web scraping techniques.

---

### Run this first: a practice API on your own machine

Public APIs go down, change their shape, rate limit a classroom of 30 people hitting
them at once, and need internet you may not have. So this cell starts a **small API
server inside this notebook**, on your own machine.

It serves paginated JSON, a flaky endpoint, a rate-limited one, a protected one and
an HTML page to scrape. Every example below works offline, and the exact same
`requests` code works against a real API.

Run it once. Re-running is safe.

In [ ]:
import json, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

CHARACTERS = [
    {"id": i,
     "name": name,
     "status": status,
     "species": "Human" if i % 3 else "Alien",
     "origin": {"name": origin, "url": f"http://localhost/location/{i}"},
     "episode": [f"http://localhost/episode/{e}" for e in range(1, (i % 4) + 2)]}
    for i, (name, status, origin) in enumerate([
        ("Rick Sanchez", "Alive", "Earth C-137"), ("Morty Smith", "Alive", "Earth C-137"),
        ("Summer Smith", "Alive", "Earth"), ("Beth Smith", "Alive", "Earth"),
        ("Jerry Smith", "Alive", "Earth"), ("Abadango Cluster", "unknown", "Abadango"),
        ("Abradolf Lincler", "unknown", "Earth"), ("Adjudicator Rick", "Dead", "unknown"),
        ("Agency Director", "Dead", "Earth"), ("Alan Rails", "Dead", "unknown"),
        ("Albert Einstein", "Dead", "Earth"), ("Alexander", "Dead", "Egypt"),
        ("Alien Googah", "unknown", "unknown"), ("Alien Morty", "unknown", "unknown"),
        ("Alien Rick", "unknown", "unknown"), ("Amish Cyborg", "Dead", "Earth"),
        ("Annie", "Alive", "Earth"), ("Antenna Morty", "Alive", "unknown"),
        ("Antenna Rick", "unknown", "unknown"), ("Ants in my Eyes Johnson", "unknown", "Earth"),
    ], start=1)
]

SHOP_HTML = '''<!doctype html>
<html>
<head><title>DEC Practice Shop</title></head>
<body>
  <h1 id="page-title">Today's Products</h1>
  <p class="intro">A small page to practise scraping.</p>

  <div class="product" data-sku="LP-001">
    <h2 class="name">Laptop Pro</h2>
    <span class="price">450000</span>
    <span class="stock in-stock">In stock</span>
    <a href="/product/lp-001">details</a>
  </div>
  <div class="product" data-sku="PH-002">
    <h2 class="name">Phone X</h2>
    <span class="price">180000</span>
    <span class="stock out-of-stock">Out of stock</span>
    <a href="/product/ph-002">details</a>
  </div>
  <div class="product" data-sku="MS-003">
    <h2 class="name">Wireless Mouse</h2>
    <span class="price">8000</span>
    <span class="stock in-stock">In stock</span>
    <a href="/product/ms-003">details</a>
  </div>

  <table id="sales">
    <thead><tr><th>Month</th><th>Units</th><th>Revenue</th></tr></thead>
    <tbody>
      <tr><td>JAN</td><td>340</td><td>1200000</td></tr>
      <tr><td>FEB</td><td>318</td><td>980000</td></tr>
      <tr><td>MAR</td><td>362</td><td>1450000</td></tr>
    </tbody>
  </table>

  <ul class="links">
    <li><a href="/about">About</a></li>
    <li><a href="https://example.com/external">External site</a></li>
    <li><a href="/contact">Contact</a></li>
  </ul>
</body>
</html>'''

ROBOTS = "User-agent: *\nDisallow: /admin\nDisallow: /checkout\nCrawl-delay: 1\n"

_state = {"flaky_calls": 0, "limited_calls": 0}
PAGE_SIZE = 6


class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass                                    # keep the notebook output clean

    def _send(self, code, payload, content_type="application/json", extra_headers=None):
        body = payload if isinstance(payload, bytes) else json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        for key, value in (extra_headers or {}).items():
            self.send_header(key, value)
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        parsed = urlparse(self.path)
        query = parse_qs(parsed.query)
        path = parsed.path

        if path == "/api/character":
            page = int(query.get("page", ["1"])[0])
            status_filter = query.get("status", [None])[0]
            rows = CHARACTERS
            if status_filter:
                rows = [c for c in rows if c["status"].lower() == status_filter.lower()]
            pages = max(1, -(-len(rows) // PAGE_SIZE))
            if page < 1 or page > pages:
                return self._send(404, {"error": "There is nothing here"})
            start = (page - 1) * PAGE_SIZE
            chunk = rows[start:start + PAGE_SIZE]
            base = f"{BASE_URL}/api/character"
            return self._send(200, {
                "info": {
                    "count": len(rows),
                    "pages": pages,
                    "next": f"{base}?page={page + 1}" if page < pages else None,
                    "prev": f"{base}?page={page - 1}" if page > 1 else None,
                },
                "results": chunk,
            })

        if path == "/api/slow":
            time.sleep(float(query.get("seconds", ["2"])[0]))
            return self._send(200, {"ok": True})

        if path == "/api/flaky":
            _state["flaky_calls"] += 1
            if _state["flaky_calls"] % 3 != 0:
                return self._send(503, {"error": "service unavailable"})
            return self._send(200, {"ok": True, "call": _state["flaky_calls"]})

        if path == "/api/limited":
            _state["limited_calls"] += 1
            if _state["limited_calls"] % 4 != 0:
                return self._send(429, {"error": "too many requests"},
                                  extra_headers={"Retry-After": "1"})
            return self._send(200, {"ok": True, "call": _state["limited_calls"]})

        if path == "/api/protected":
            if self.headers.get("X-API-Key") != "dec-secret-key":
                return self._send(401, {"error": "missing or invalid API key"})
            return self._send(200, {"secret": "you are authorised", "plan": "pro"})

        if path == "/api/broken":
            return self._send(200, b"this is not json at all", content_type="text/plain")

        if path == "/shop.html":
            return self._send(200, SHOP_HTML.encode(), content_type="text/html")

        if path == "/robots.txt":
            return self._send(200, ROBOTS.encode(), content_type="text/plain")

        return self._send(404, {"error": "not found"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        raw = self.rfile.read(length)
        try:
            payload = json.loads(raw) if raw else {}
        except json.JSONDecodeError:
            return self._send(400, {"error": "body was not valid json"})
        if urlparse(self.path).path != "/api/orders":
            return self._send(404, {"error": "not found"})
        if "item" not in payload:
            return self._send(422, {"error": "field 'item' is required"})
        return self._send(201, {"created": True, "order_id": 9001, "echo": payload})


class QuietServer(ThreadingHTTPServer):
    """Like ThreadingHTTPServer, but silent when a client hangs up early."""
    daemon_threads = True

    def handle_error(self, request, client_address):
        pass                                    # a client timing out is expected here


def start_server():
    """Start the practice API on a free port. Returns the base URL."""
    server = QuietServer(("127.0.0.1", 0), Handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server, f"http://127.0.0.1:{server.server_address[1]}"


if "_server" in dir() and _server:                     # re-run safe
    _server.shutdown()

_server, BASE_URL = start_server()
time.sleep(0.3)

print(f"practice API running at {BASE_URL}")
print()
print("endpoints:")
for line in [
    "GET  /api/character?page=N    paginated JSON, 20 records over 4 pages",
    "GET  /api/slow?seconds=N      takes its time, for timeout practice",
    "GET  /api/flaky               fails twice, succeeds on the third call",
    "GET  /api/limited             429s three times, succeeds on the fourth",
    "GET  /api/protected           401 unless you send X-API-Key",
    "GET  /api/broken              200 OK, but the body is not JSON",
    "GET  /shop.html               a page to scrape",
    "GET  /robots.txt              the rules of the site",
    "POST /api/orders              accepts JSON, 422 if 'item' is missing",
]:
    print(f"  {line}")

In [ ]:
# Confirm it works
import requests

response = requests.get(f"{BASE_URL}/api/character", timeout=5)
print(response.status_code)
print(response.json()["info"])

#### `GET`
##### Purpose: Used to request data from a specified resource.
- `Data Transmission`: Sends data appended to the URL as query parameters (e.g., example.com/page?name=value).
- `Security`: Less secure because the data is visible in the URL, which can be logged or cached.
- `Data Size`: Limited in size due to URL length restrictions.
- `Idempotency`: Safe and idempotent, meaning multiple identical requests have the same effect as a single request.
- `Caching`: Responses can be cached by browsers and intermediate proxies.

`Use Case`: Suitable for retrieving data without causing any side effects on the server.


#### `POST`
##### Purpose: Used to send data to a server to create or update a resource.
- `Data Transmission`: Sends data in the request body, not visible in the URL.
- `Security`: More secure than GET as data isn't exposed in the URL, but still not fully secure without HTTPS.
- `Data Size`: No specific size limitation (depends on server settings), allowing for larger data transmission.
- `Idempotency`: Not idempotent; multiple identical requests can create or modify multiple resources or have different effects.
- `Caching`: Responses are not typically cached.

`Use Case`: Suitable for submitting forms, uploading files, or performing operations that change the server's state.

In summary, use `GET` for retrieving data and `POST` for sending data that modifies or updates server resources.

## 1. Working with APIs
APIs allow you to communicate with web services and retrieve data. You can use Python's `requests` library to send HTTP requests to APIs.

In [ ]:
#https://rickandmortyapi.com/documentation/

In [ ]:
import requests

# Example: Sending a GET request to a public API
url = "https://rickandmortyapi.com/api"

response = requests.get(url)

# Check the status code (200 means success)
if response.status_code == 200:
    data = response.json()  # Parse JSON response
    print(data)
else:
    print("Error:", response.status_code)


---

## Practice: the response object

`requests.get()` gives you back a `Response`. Learn its parts once and every API in
the world becomes the same job.

In [ ]:
import requests

response = requests.get(f"{BASE_URL}/api/character", params={"page": 2}, timeout=10)

print(f"status_code : {response.status_code}")
print(f"ok          : {response.ok}")
print(f"url         : {response.url}")
print(f"elapsed     : {response.elapsed.total_seconds():.3f}s")
print(f"content-type: {response.headers['Content-Type']}")
print(f"length      : {len(response.content)} bytes")
print()
print(f"text is a {type(response.text).__name__}, json() gives a {type(response.json()).__name__}")

Note `params={"page": 2}` rather than gluing `?page=2` onto the URL yourself.
`requests` builds the query string and escapes anything that needs escaping, which
matters the moment a value contains a space or an ampersand.

In [ ]:
# params handles the escaping for you
messy = requests.get(f"{BASE_URL}/api/character",
                     params={"status": "Alive", "note": "a & b = c"}, timeout=10)
print(messy.url)

### Always pass a timeout

Without one, `requests` waits forever. A hung request in a scheduled job means the
job never finishes, never alerts, and quietly stops producing data until somebody
notices next week.

In [ ]:
# What a timeout looks like
try:
    requests.get(f"{BASE_URL}/api/slow", params={"seconds": 3}, timeout=1)
except requests.exceptions.Timeout:
    print("Timeout after 1 second, exactly as asked")

response = requests.get(f"{BASE_URL}/api/slow", params={"seconds": 1}, timeout=5)
print(f"with a longer timeout: {response.status_code} in "
      f"{response.elapsed.total_seconds():.2f}s")

You can also give it two numbers: `timeout=(3, 30)` means "3 seconds to connect,
30 seconds to read the response". Useful when the server is quick to answer but slow
to finish sending.

In [ ]:
# Checking the status. Three ways, worst to best.

# 1. Ignore it entirely. The bug you will write once.
data = requests.get(f"{BASE_URL}/api/character", params={"page": 99}, timeout=5).json()
print(f"page 99 gave: {data}")
print("   no crash, no data, and your loop happily carries on with nothing")

# 2. Check it yourself
response = requests.get(f"{BASE_URL}/api/character", params={"page": 99}, timeout=5)
if response.status_code == 200:
    print("ok")
else:
    print(f"   checked manually: HTTP {response.status_code}")

# 3. Let requests raise
try:
    response.raise_for_status()
except requests.exceptions.HTTPError as e:
    print(f"   raise_for_status: {e}")

`raise_for_status()` turns any 4xx or 5xx into an exception you can catch. It is one
line and it converts a silent wrong answer into a loud, catchable failure.

Put it after every request unless you have a specific reason not to.

In [ ]:
# The status codes you will actually handle, and what to do about each
HANDLING = [
    (200, "OK",                  "carry on"),
    (201, "Created",             "your POST worked"),
    (204, "No Content",          "worked, nothing to parse. do not call .json()"),
    (301, "Moved",               "requests follows it for you"),
    (400, "Bad Request",         "your request is wrong. fix it, do not retry"),
    (401, "Unauthorized",        "missing or bad credentials. do not retry"),
    (403, "Forbidden",           "authenticated but not allowed. do not retry"),
    (404, "Not Found",           "wrong URL, or the record is gone"),
    (422, "Unprocessable",       "the shape is right, a value is not"),
    (429, "Too Many Requests",   "slow down. wait, then retry"),
    (500, "Server Error",        "their fault. retry with backoff"),
    (502, "Bad Gateway",         "their fault. retry with backoff"),
    (503, "Service Unavailable", "their fault. retry with backoff"),
    (504, "Gateway Timeout",     "their fault. retry with backoff"),
]

print(f"{'code':<5} {'meaning':<21} what to do")
print("-" * 70)
for code, meaning, action in HANDLING:
    print(f"{code:<5} {meaning:<21} {action}")

print()
print("The rule: 4xx means YOU did something wrong, so retrying changes nothing.")
print("          5xx means THEY did, so retrying often works. 429 is the exception:")
print("          it is a 4xx that you should absolutely retry, after waiting.")

In [ ]:
# The exceptions requests can raise, and catching them properly
import requests

def fetch(url, **kwargs):
    """GET a URL, returning parsed JSON or None with a clear message."""
    try:
        response = requests.get(url, timeout=kwargs.pop("timeout", 5), **kwargs)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        print("  timed out. the server is slow or hanging")
    except requests.exceptions.ConnectionError:
        print("  could not connect. DNS, network, or the server is down")
    except requests.exceptions.HTTPError as e:
        print(f"  HTTP error: {e.response.status_code}")
    except requests.exceptions.JSONDecodeError:
        print("  the response was not JSON")
    except requests.exceptions.RequestException as e:
        print(f"  something else: {type(e).__name__}")
    return None


for label, url in [
    ("good page",      f"{BASE_URL}/api/character?page=1"),
    ("missing page",   f"{BASE_URL}/api/character?page=99"),
    ("not json",       f"{BASE_URL}/api/broken"),
    ("nothing there",  "http://127.0.0.1:9/nope"),
]:
    print(label)
    result = fetch(url)
    if result:
        print(f"  got {len(result.get('results', []))} record(s)")

`requests.exceptions.RequestException` is the parent of all of them, so it goes
last as the safety net. Same shape as `except Exception` going last in the errors
notebook.

### Your turn: tasks

**Task 1.** Request page 3 of `/api/character` using `params=`, print the status code,
the URL requests built, and how many results came back.

**Task 2.** Call `/api/slow?seconds=2` with `timeout=1` and catch the `Timeout`, then
again with `timeout=5` and print how long it took.

**Task 3.** Request page 99 and use `raise_for_status()` to turn it into an exception
you catch and report.

**Task 4.** Call `/api/broken` and catch the error from trying to parse it as JSON.

**DE Task 5.** Write `safe_get(url, params=None)` that returns `(data, error)`, where
`error` is `None` on success and a short string otherwise. Handle timeout, connection
error, HTTP error and bad JSON separately. Test it against four different endpoints.

In [ ]:
# Task 1 answer
import requests


In [ ]:
# Task 2 answer


In [ ]:
# Task 3 answer


In [ ]:
# Task 4 answer


In [ ]:
# DE Task 5 answer


### HTTP Status Codes

#### 1xx: Informational
- **100 Continue**: The initial part of a request has been received and the client can continue with the request or ignore if it is already finished.
- **101 Switching Protocols**: The server is switching protocols as requested by the client (e.g., switching from HTTP/1.1 to HTTP/2).

#### 2xx: Success
- **200 OK**: The request has succeeded. The information returned depends on the request method (e.g., GET, POST).
- **201 Created**: The request has been fulfilled and resulted in a new resource being created.
- **202 Accepted**: The request has been accepted for processing, but the processing is not complete.
- **204 No Content**: The server successfully processed the request, but there's no content to send back.
- **205 Reset Content**: The server successfully processed the request, but asks the client to reset the document view.

#### 3xx: Redirection
- **300 Multiple Choices**: There are multiple options for the resource the client may follow.
- **301 Moved Permanently**: The resource has been moved permanently to a new URL.
- **302 Found**: The resource has been temporarily moved to a different URL.
- **304 Not Modified**: The resource has not been modified since the last request, so the client can use the cached version.
- **307 Temporary Redirect**: The request should be repeated with another URL, but future requests should still use the original URL.
- **308 Permanent Redirect**: The resource has been permanently moved to a new URL, and all future requests should use the new URL.

#### 4xx: Client Error
- **400 Bad Request**: The server could not understand the request due to invalid syntax.
- **401 Unauthorized**: Authentication is required and has failed or has not yet been provided.
- **403 Forbidden**: The server understood the request but refuses to authorize it.
- **404 Not Found**: The server can't find the requested resource.
- **405 Method Not Allowed**: The request method is known by the server but has been disabled and cannot be used for the requested resource.
- **409 Conflict**: The request could not be completed due to a conflict with the current state of the resource.
- **410 Gone**: The requested resource is no longer available and will not be available again.
- **415 Unsupported Media Type**: The server refuses to accept the request because the payload format is in an unsupported format.
- **429 Too Many Requests**: The user has sent too many requests in a given amount of time ("rate limiting").

#### 5xx: Server Error
- **500 Internal Server Error**: The server encountered an unexpected condition that prevented it from fulfilling the request.
- **501 Not Implemented**: The server does not support the functionality required to fulfill the request.
- **502 Bad Gateway**: The server, while acting as a gateway or proxy, received an invalid response from the upstream server.
- **503 Service Unavailable**: The server is not ready to handle the request, often due to maintenance or overload.
- **504 Gateway Timeout**: The server, while acting as a gateway or proxy, did not receive a timely response from the upstream server.
- **505 HTTP Version Not Supported**: The server does not support the HTTP protocol version used in the request.


In [ ]:
import requests
base_url = 'https://rickandmortyapi.com/api/'
endpoint = 'character'

req= requests.get(base_url + endpoint)

In [ ]:
req.status_code

In [ ]:
req.json()

In [ ]:
data = req.json()

In [ ]:
data['results']

In [ ]:
import pandas as pd
pd.DataFrame(data['results'])

### Pagination

In [ ]:
import requests
base_url = 'https://rickandmortyapi.com/api/'
endpoint = 'character'

req= requests.get(base_url + endpoint)
data = req.json()
pages= data['info']['pages']

for page in range(1,pages+1):
    req= requests.get(base_url + endpoint + f'?pages={page}')
    print(req.status_code)
    


In [ ]:
import requests

base_url = 'https://rickandmortyapi.com/api/'
endpoint = 'character'

req = requests.get(base_url + endpoint)
data = req.json()

# Loop through all pages using the 'next' field
while True:
    print(req.status_code)
    
    # Get the next page URL
    next_url = data['info']['next']
    
    if not next_url:
        break  # No more pages, exit the loop
    
    req = requests.get(next_url)
    data = req.json()

    

---

## Practice: pagination, retries and rate limits

### A bug worth staring at

There is a pagination loop earlier in this notebook:

```python
for page in range(1, pages+1):
    req = requests.get(base_url + endpoint + f'?pages={page}')
    print(req.status_code)
```

Two problems.

**`?pages=` should be `?page=`.** The API does not know what `pages` means, so it
ignores it and returns page 1 every single time. You get 200 OK forty-two times and
forty-two copies of the same six records. Nothing errors. Nothing warns you.

**Nothing is collected.** The loop prints a status code and throws the data away.

Here is what it looks like when the parameter name is wrong.

In [ ]:
import requests

# The buggy version: wrong parameter name
names_seen = []
for page in range(1, 5):
    r = requests.get(f"{BASE_URL}/api/character", params={"pages": page}, timeout=5)
    first_name = r.json()["results"][0]["name"]
    names_seen.append(first_name)
    print(f"  page {page}: status {r.status_code}, first record {first_name!r}")

print()
print(f"distinct first records: {len(set(names_seen))}. Should be 4.")
print("Every request returned page 1. The typo cost you three quarters of the data.")

In [ ]:
# The fixed version, which also keeps the data
all_characters = []

first = requests.get(f"{BASE_URL}/api/character", timeout=5)
first.raise_for_status()
payload = first.json()
total_pages = payload["info"]["pages"]
all_characters.extend(payload["results"])

for page in range(2, total_pages + 1):
    response = requests.get(f"{BASE_URL}/api/character", params={"page": page}, timeout=5)
    response.raise_for_status()
    all_characters.extend(response.json()["results"])
    print(f"  page {page}/{total_pages}: {len(response.json()['results'])} record(s)")

print()
print(f"collected {len(all_characters)} records, "
      f"{len({c['id'] for c in all_characters})} of them unique")

**Always check the unique count.** `len(all_characters)` being 20 proves nothing
if the same 6 records arrived four times. Counting distinct ids is a two-second check
that catches a whole class of pagination bug.

In [ ]:
# Better: follow the "next" link instead of counting pages yourself.
# The API tells you when it is finished, so you cannot get the arithmetic wrong.

collected = []
url = f"{BASE_URL}/api/character"
page_count = 0

while url:
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    payload = response.json()

    collected.extend(payload["results"])
    page_count += 1
    url = payload["info"]["next"]          # None on the last page, which ends the loop

print(f"{page_count} page(s), {len(collected)} record(s), "
      f"{len({c['id'] for c in collected})} unique")

This is the pattern to default to. It survives the API changing its page size, and
there is no off-by-one to get wrong.

**Always put a safety limit on it though.** A server bug that returns itself as
`next` turns this into an infinite loop that hammers someone's API until they block
you.

In [ ]:
# The same loop, with a guard
def fetch_all(url, max_pages=50):
    """Follow next links, collecting results. Stops at max_pages whatever happens."""
    collected = []
    pages = 0
    seen_urls = set()

    while url and pages < max_pages:
        if url in seen_urls:
            print(f"  the API sent us back to {url}. stopping")
            break
        seen_urls.add(url)

        response = requests.get(url, timeout=10)
        response.raise_for_status()
        payload = response.json()

        collected.extend(payload["results"])
        pages += 1
        url = payload["info"]["next"]

    if pages >= max_pages:
        print(f"  hit the {max_pages} page limit. is that expected?")
    return collected, pages


rows, pages = fetch_all(f"{BASE_URL}/api/character")
print(f"{len(rows)} records over {pages} page(s)")

### Retries with backoff

A 500 or a dropped connection is usually temporary. Retrying immediately is rude and
often useless, because the server is still struggling. Wait, and wait longer each
time.

In [ ]:
import time

def get_with_retry(url, max_attempts=5, base_delay=0.2, **kwargs):
    """GET with exponential backoff. Retries 5xx and connection errors only."""
    delay = base_delay

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(url, timeout=kwargs.pop("timeout", 10), **kwargs)

            if response.status_code < 400:
                print(f"  attempt {attempt}: {response.status_code} ok")
                return response

            if 400 <= response.status_code < 500 and response.status_code != 429:
                print(f"  attempt {attempt}: {response.status_code}, our fault. not retrying")
                response.raise_for_status()

            print(f"  attempt {attempt}: {response.status_code}, retrying in {delay:.1f}s")

        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            print(f"  attempt {attempt}: {type(e).__name__}, retrying in {delay:.1f}s")

        if attempt < max_attempts:
            time.sleep(delay)
            delay *= 2                                  # 0.2, 0.4, 0.8, 1.6 ...

    raise RuntimeError(f"gave up on {url} after {max_attempts} attempts")


print("the flaky endpoint fails twice then works:")
result = get_with_retry(f"{BASE_URL}/api/flaky")
print(f"  -> {result.json()}")

Notice it does **not** retry a 400 or a 401. Retrying those wastes time and can get
you rate limited or blocked: a bad API key will still be a bad API key on the fifth
attempt. Only retry what might genuinely succeed next time.

In [ ]:
# 429 and Retry-After: the server is telling you exactly how long to wait
def get_respecting_rate_limit(url, max_attempts=6, **kwargs):
    """GET, honouring the Retry-After header when the server sends a 429."""
    for attempt in range(1, max_attempts + 1):
        response = requests.get(url, timeout=10, **kwargs)

        if response.status_code != 429:
            print(f"  attempt {attempt}: {response.status_code}")
            return response

        wait = float(response.headers.get("Retry-After", 1))
        print(f"  attempt {attempt}: 429, server says wait {wait}s")
        time.sleep(wait)

    raise RuntimeError("still rate limited after several attempts")


result = get_respecting_rate_limit(f"{BASE_URL}/api/limited")
print(f"  -> {result.json()}")

Read `Retry-After` rather than guessing. When the server tells you how long to
wait, waiting less is how you get blocked and waiting more is wasted time.

In [ ]:
# Sessions: reuse the connection, set headers once
session = requests.Session()
session.headers.update({
    "User-Agent": "DEC-Class-Bot/1.0 (teaching example)",
    "Accept": "application/json",
})

start = time.perf_counter()
for page in range(1, 4):
    session.get(f"{BASE_URL}/api/character", params={"page": page}, timeout=5)
with_session = time.perf_counter() - start

start = time.perf_counter()
for page in range(1, 4):
    requests.get(f"{BASE_URL}/api/character", params={"page": page}, timeout=5)
without_session = time.perf_counter() - start

print(f"with a session:    {with_session * 1000:.1f} ms")
print(f"without:           {without_session * 1000:.1f} ms")
print()
print("On localhost the difference is small. Over the internet, where every new")
print("connection means a DNS lookup and a TLS handshake, a Session across 1,000")
print("requests saves minutes.")

In [ ]:
# Authentication: the key comes from the environment, never from the notebook
import os

response = session.get(f"{BASE_URL}/api/protected", timeout=5)
print(f"without a key: {response.status_code} {response.json()}")

os.environ.setdefault("DEC_API_KEY", "dec-secret-key")      # normally set in your shell

response = session.get(
    f"{BASE_URL}/api/protected",
    headers={"X-API-Key": os.environ["DEC_API_KEY"]},
    timeout=5,
)
print(f"with a key:    {response.status_code} {response.json()}")

Three ways APIs ask for credentials, all of which come from the environment:

```python
headers={"X-API-Key": os.environ["API_KEY"]}                  # custom header
headers={"Authorization": f"Bearer {os.environ['TOKEN']}"}    # bearer token
auth=(os.environ["USER"], os.environ["PASSWORD"])             # basic auth
```

An API key in a notebook is the same mistake as a database password in a notebook,
with the same consequence: once it has been shared, it is compromised.

In [ ]:
# POST: sending data
new_order = {"item": "Laptop Pro", "quantity": 1, "customer_id": 101}

response = session.post(f"{BASE_URL}/api/orders", json=new_order, timeout=5)
print(f"{response.status_code}: {response.json()}")

print()
bad = session.post(f"{BASE_URL}/api/orders", json={"quantity": 1}, timeout=5)
print(f"{bad.status_code}: {bad.json()}")

`json=` sets the `Content-Type` header and serialises the dict for you.
`data=` sends form-encoded values instead. Sending a dict with `data=` when the API
wanted JSON produces a 400 that takes an hour to diagnose.

### Your turn: tasks

**Task 1.** Fetch all pages of `/api/character` by following the `next` link, and
print both the total records and the distinct ids.

**Task 2.** Do the same using `range(1, pages+1)` and the **correct** parameter name.
Confirm both give the same distinct count.

**Task 3.** Call `/api/flaky` in a loop with exponential backoff until it succeeds,
printing each attempt and the wait.

**Task 4.** Call `/api/limited` and honour the `Retry-After` header.

**DE Task 5.** Write `fetch_all(url, max_pages=50)` that follows next links, refuses
to visit the same URL twice, stops at the limit, and returns
`(records, pages_fetched)`.

**DE Task 6.** Write `api_get(session, url, **kwargs)` that combines everything:
timeout, `raise_for_status`, retries with backoff on 5xx and connection errors,
`Retry-After` on 429, and no retry on other 4xx.

**DE Task 7.** Fetch every character with `status=Alive` using the `status` query
parameter, and report how many there are compared with the unfiltered total.

In [ ]:
# Task 1 answer


In [ ]:
# Task 2 answer


In [ ]:
# Task 3 answer


In [ ]:
# Task 4 answer


In [ ]:
# DE Task 5 answer


In [ ]:
# DE Task 6 answer


In [ ]:
# DE Task 7 answer


---

## Practice: turning nested JSON into a table

API responses are nested. Tables are flat. Closing that gap is most of the work in
an API extraction job.

In [ ]:
import requests, pandas as pd

response = requests.get(f"{BASE_URL}/api/character", timeout=5)
records = response.json()["results"]

print(records[0])

In [ ]:
# The naive version. Look at the origin and episode columns.
naive = pd.DataFrame(records)
print(naive.dtypes)
naive.head(3)

`origin` is a whole dictionary sitting in a cell, and `episode` is a list. You
cannot group by a dictionary, filter on it, or write it to a warehouse column.
Nested data has to be flattened.

In [ ]:
# json_normalize flattens the nested dictionaries into their own columns
flat = pd.json_normalize(records)
print(list(flat.columns))
flat[["id", "name", "status", "origin.name"]].head(3)

In [ ]:
# Control the separator, and choose which columns you keep
flat = pd.json_normalize(records, sep="_")
flat = flat.rename(columns={"origin_name": "origin", "origin_url": "origin_url"})

wanted = ["id", "name", "status", "species", "origin"]
table = flat[wanted].copy()
table["episode_count"] = [len(r["episode"]) for r in records]

table.head()

The `episode` list needed a different treatment: a list of unknown length cannot
become a column. You either count it (as here), keep it as a JSON string, or
**explode** it into its own table, which is what a real warehouse does.

In [ ]:
# Exploding a list into a child table: one row per character per episode
exploded = pd.DataFrame([
    {"character_id": r["id"], "episode_url": url}
    for r in records
    for url in r["episode"]
])

print(f"{len(records)} characters -> {len(exploded)} character-episode rows")
exploded.head()

That nested loop in a comprehension is the flattening pattern: outer loop over
records, inner loop over the list inside each one. You now have a parent table and a
child table, which is exactly how this data belongs in a warehouse.

In [ ]:
# Writing results out as you go, so a crash on page 40 does not lose pages 1 to 39
import json, os

def fetch_to_jsonl(start_url, path, max_pages=50):
    """Page through an API, appending each record to a JSONL file as it arrives."""
    url = start_url
    pages = written = 0

    with open(path, "w", encoding="utf-8") as f:
        while url and pages < max_pages:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            payload = response.json()

            for record in payload["results"]:
                f.write(json.dumps(record) + "\n")
                written += 1

            pages += 1
            url = payload["info"]["next"]

    return pages, written


pages, written = fetch_to_jsonl(f"{BASE_URL}/api/character", "characters.jsonl")
print(f"{pages} page(s), {written} record(s), file is {os.path.getsize('characters.jsonl')} bytes")

with open("characters.jsonl", encoding="utf-8") as f:
    print(json.loads(f.readline())["name"])

In [ ]:
# Resuming: skip what you already have
def fetch_resumable(start_url, path, max_pages=50):
    """Append only records whose id is not already in the file."""
    seen = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                seen.add(json.loads(line)["id"])

    url, pages, added = start_url, 0, 0
    with open(path, "a", encoding="utf-8") as f:
        while url and pages < max_pages:
            payload = requests.get(url, timeout=10).json()
            for record in payload["results"]:
                if record["id"] not in seen:
                    f.write(json.dumps(record) + "\n")
                    seen.add(record["id"])
                    added += 1
            pages += 1
            url = payload["info"]["next"]

    return added, len(seen)


added, total = fetch_resumable(f"{BASE_URL}/api/character", "characters.jsonl")
print(f"run 2: added {added} new record(s), {total} in total")

added, total = fetch_resumable(f"{BASE_URL}/api/character", "characters.jsonl")
print(f"run 3: added {added} new record(s), {total} in total")

Zero added on re-runs. That is the same idempotency idea as the database notebook,
and it is what lets you restart a failed extraction without re-downloading everything
or creating duplicates.

### Your turn: tasks

**Task 1.** Fetch page 1 and build a DataFrame with `pd.DataFrame()`. Print the dtypes
and identify which columns are not flat.

**Task 2.** Use `pd.json_normalize` on the same records and print the new column names.

**Task 3.** Build a clean table with `id`, `name`, `status`, `species`, `origin` and
`episode_count`.

**DE Task 4.** Fetch every page, flatten, and print a `value_counts()` of `status` and
of `species`.

**DE Task 5.** Build the exploded character-to-episode child table for all pages, and
report the row count of each table.

**DE Task 6.** Write `fetch_to_jsonl(start_url, path)` that streams every record to a
JSONL file as it pages, then read the file back and confirm the count matches.

**DE Task 7.** Make it resumable: re-running must add zero new records. Prove it by
running it three times.

In [ ]:
# Task 1 answer
import pandas as pd


In [ ]:
# Task 2 answer


In [ ]:
# Task 3 answer


In [ ]:
# DE Task 4 answer


In [ ]:
# DE Task 5 answer


In [ ]:
# DE Task 6 answer


In [ ]:
# DE Task 7 answer


## 2. Web Scraping with BeautifulSoup
Web scraping involves extracting data from web pages. `BeautifulSoup` is a Python library used for parsing HTML documents and extracting specific elements.

In [ ]:
from bs4 import BeautifulSoup
import requests

# Example: Scraping a webpage
url = "https://example.com"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# Extract title of the webpage
title = soup.title.string
print("Page Title:", title)

# Extract all the links from the page
links = soup.find_all('a')
for link in links:
    print(link.get('href'))


---

## Practice: scraping with BeautifulSoup

### Before you write a line of code

An API is a front door. Scraping is a window. The window is often open, but there are
still rules.

1. **Is there an API?** Check first. It will be faster, stabler and allowed.
2. **Read `robots.txt`.** It tells you which paths the site asks you not to crawl.
3. **Read the terms of service.** `robots.txt` is a request; the ToS is a contract.
4. **Identify yourself** with a real `User-Agent`, ideally with contact details.
5. **Rate limit yourself.** One request a second is polite. Twenty a second is an
   attack as far as their monitoring is concerned.
6. **Cache what you fetch.** Do not re-download the same page while developing.
7. **Personal data is regulated** whether or not it was easy to collect. GDPR and
   NDPR do not care that the page was public.

None of this is legal advice. It is the difference between a scraper that runs for
years and one that gets your office IP blocked on day two.

In [ ]:
import requests

robots = requests.get(f"{BASE_URL}/robots.txt", timeout=5)
print(robots.text)

print("This site asks you to stay out of /admin and /checkout,")
print("and to wait 1 second between requests.")

In [ ]:
# Python can parse robots.txt for you
from urllib.robotparser import RobotFileParser

parser = RobotFileParser()
parser.set_url(f"{BASE_URL}/robots.txt")
parser.read()

for path in ["/shop.html", "/admin", "/checkout", "/about"]:
    allowed = parser.can_fetch("DEC-Class-Bot", f"{BASE_URL}{path}")
    print(f"{path:<14} {'allowed' if allowed else 'DISALLOWED'}")

### Parsing the page

In [ ]:
from bs4 import BeautifulSoup
import requests

response = requests.get(f"{BASE_URL}/shop.html", timeout=5)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print(f"title: {soup.title.string}")
print(f"h1:    {soup.h1.text}")

**Which parser?** `"html.parser"` is built in and fine for clean pages.
`"lxml"` is faster and much more forgiving of broken HTML, which describes most real
pages. Install it with `pip install lxml` and pass `"lxml"` instead.

Note also `response.text` (a string, decoded) versus `response.content` (raw bytes).
Pass `.content` when the page declares an unusual encoding and `.text` looks wrong.

In [ ]:
# find and find_all: the two you will use most
first_product = soup.find("div", class_="product")
print("first product block:")
print(first_product.prettify()[:300])

print()
all_products = soup.find_all("div", class_="product")
print(f"{len(all_products)} product block(s) on the page")

`class_` has a trailing underscore because `class` is a reserved word in Python.
Everyone forgets this once.

In [ ]:
# Pulling the fields out of each block
products = []

for block in soup.find_all("div", class_="product"):
    products.append({
        "sku": block.get("data-sku"),                       # an attribute
        "name": block.find("h2", class_="name").text.strip(),
        "price": int(block.find("span", class_="price").text.strip()),
        "in_stock": "in-stock" in block.find("span", class_="stock").get("class"),
        "url": block.find("a").get("href"),
    })

for product in products:
    print(product)

In [ ]:
# CSS selectors: often shorter, and the same syntax you use in the browser console
print([tag.text for tag in soup.select("div.product h2.name")])
print([tag.text for tag in soup.select("span.price")])
print([tag.get("href") for tag in soup.select("ul.links a")])
print(soup.select_one("#page-title").text)
print([tag.text for tag in soup.select("div.product .stock.in-stock")])

| You want | `find` style | CSS selector |
|---|---|---|
| By tag | `soup.find_all("a")` | `soup.select("a")` |
| By class | `soup.find_all("div", class_="product")` | `soup.select("div.product")` |
| By id | `soup.find(id="sales")` | `soup.select_one("#sales")` |
| Nested | two calls | `soup.select("div.product h2")` |
| By attribute | `soup.find_all("div", attrs={"data-sku": "LP-001"})` | `soup.select('[data-sku="LP-001"]')` |

Use whichever reads better. `select` tends to win once you are going more than one
level deep.

In [ ]:
# Getting text, and the trap in it
block = soup.find("div", class_="product")

print(f".text        : {block.text!r}")
print()
print(f".get_text(strip=True, separator='|'): {block.get_text(strip=True, separator='|')!r}")

`.text` on a container glues together every descendant's text, whitespace and all.
When you want one field, find that specific tag first. When you want everything,
`get_text(strip=True, separator=" ")` gives you something readable.

In [ ]:
# Links: relative versus absolute
from urllib.parse import urljoin

for link in soup.select("ul.links a"):
    href = link.get("href")
    print(f"{link.text:<14} raw: {href:<32} absolute: {urljoin(BASE_URL, href)}")

`urljoin` handles relative paths correctly, including the awkward cases. Building
URLs with `BASE_URL + href` breaks the moment one of them already has a slash, or the
href is already absolute.

In [ ]:
# Tables: let pandas do it
import pandas as pd
from io import StringIO

tables = pd.read_html(StringIO(response.text))
print(f"{len(tables)} table(s) found")

sales = tables[0]
print(sales)
print()
print(sales.dtypes)

`pd.read_html` finds every `<table>` on a page and returns a list of DataFrames.
For tabular data it replaces twenty lines of BeautifulSoup. Wrap the HTML in
`StringIO` to avoid a deprecation warning about passing a bare string.

If it says "No tables found" the data is probably laid out with `div`s rather than a
real `<table>`, and you are back to BeautifulSoup.

In [ ]:
# Defensive scraping: the page WILL change, so never assume a tag is there
def safe_text(node, selector, default=""):
    """Return the text of the first match, or a default when it is missing."""
    found = node.select_one(selector)
    return found.text.strip() if found else default


def parse_product(block):
    """Parse one product block, surviving missing fields."""
    price_text = safe_text(block, "span.price")
    try:
        price = float(price_text.replace(",", ""))
    except ValueError:
        price = None

    return {
        "sku": block.get("data-sku", "unknown"),
        "name": safe_text(block, "h2.name", "unnamed"),
        "price": price,
        "rating": safe_text(block, "span.rating", "not shown"),     # never exists
        "price_raw": price_text,
    }


for block in soup.find_all("div", class_="product"):
    print(parse_product(block))

print()
print("The rating selector matches nothing, and nothing crashed.")
print("A scraper that raises AttributeError on a missing tag will break")
print("the first time the site ships a redesign, which is always.")

In [ ]:
# Being polite: a session, an honest User-Agent, and a delay between requests
import time

scraper = requests.Session()
scraper.headers.update({
    "User-Agent": "DEC-Class-Bot/1.0 (+https://dataengineeringcommunity.com; teaching)",
})

CRAWL_DELAY = 1.0          # from robots.txt

pages_to_fetch = ["/shop.html", "/shop.html", "/shop.html"]
cache = {}

for path in pages_to_fetch:
    if path in cache:
        print(f"  {path}: served from cache, no request made")
        continue

    response = scraper.get(f"{BASE_URL}{path}", timeout=10)
    cache[path] = response.text
    print(f"  {path}: fetched, {len(response.text)} bytes")
    time.sleep(CRAWL_DELAY)

print(f"\n{len(cache)} page(s) cached, 1 request made for 3 requested pages")

### What scraping cannot do

If the page is built by JavaScript after it loads, `requests` gets the empty shell
and BeautifulSoup finds nothing. Check by viewing the page source (not the inspector)
and searching for the text you want. Missing there means it was added by JavaScript.

Your options then, in order of preference:

1. **Find the API the page itself calls.** Open the network tab, reload, look for a
   JSON request. Nine times out of ten it is there, and it is far better than
   scraping the rendered page.
2. Use a browser automation tool such as Playwright or Selenium, which runs the
   JavaScript. Slower and heavier, but it works.
3. Reconsider whether you need this data.

### Your turn: tasks

**Task 1.** Fetch `/robots.txt` and print which paths are disallowed. Then use
`RobotFileParser` to check whether `/shop.html` and `/admin` are allowed.

**Task 2.** Parse `/shop.html` and print the page title and the `h1` text.

**Task 3.** Print the name and price of every product, using `find_all`.

**Task 4.** Do the same with CSS selectors and `select`.

**Task 5.** Print every link on the page as an absolute URL using `urljoin`.

**DE Task 6.** Build a DataFrame of the products with columns `sku`, `name`, `price`
and `in_stock`, and print the total value of the in-stock items.

**DE Task 7.** Extract the sales table with `pd.read_html` and print the total revenue.

**DE Task 8.** Write `scrape_products(url, session)` that is defensive: missing tags
give defaults instead of raising, prices that will not parse become `None`, and it
returns a list of dicts. Test it, then test it again on a snippet of HTML where the
price tag has been removed.

In [ ]:
# Task 1 answer


In [ ]:
# Task 2 answer


In [ ]:
# Task 3 answer


In [ ]:
# Task 4 answer


In [ ]:
# Task 5 answer


In [ ]:
# DE Task 6 answer


In [ ]:
# DE Task 7 answer


In [ ]:
# DE Task 8 answer


---

## End of class mini project: an API extraction job

Pull every record from the practice API, flatten it, validate it, and land it as
files, without anything crashing the run and without a re-run duplicating data.

### What to build

**1. `make_session()`** returns a `requests.Session` with a real `User-Agent` and an
`Accept` header set once.

**2. `api_get(session, url, **kwargs)`** does one request properly:
   - a timeout on every call
   - `raise_for_status()`
   - retry on 5xx, `ConnectionError` and `Timeout`, with the delay doubling each time
   - honour `Retry-After` on a 429
   - never retry a 400, 401, 403 or 404
   - give up after 5 attempts with a clear exception

**3. `fetch_all(session, start_url, max_pages=50)`** follows the `next` links,
refuses to visit a URL twice, and returns `(records, pages_fetched)`.

**4. `flatten(records)`** returns two DataFrames:
   - `characters`: `id`, `name`, `status`, `species`, `origin`, `episode_count`
   - `character_episodes`: `character_id`, `episode_url`, one row per pair

**5. `validate(df)`** returns `(clean, rejected)`. Reject a character when the `name`
is empty, the `id` is missing, or `status` is not one of
`{"Alive", "Dead", "unknown"}`. Keep the reason.

**6. `save(characters, episodes, rejected)`** writes `characters.csv`,
`character_episodes.csv` and `rejected.csv`, all with `index=False`, plus
`run_summary.json` holding the counts and the timestamp.

**7. `run()`** ties it together, prints a readable report, and returns the summary.

### Prove it works

- Print the total records and the **distinct** id count. They must match.
- Run it twice and show `characters.csv` has the same row count both times.
- Point `fetch_all` at `/api/flaky` once to show the retry logic firing.

### Stretch goals

8. Add a `status` filter parameter so `run(status="Alive")` fetches only those, and
   report how many that is out of the total.
9. Make it resumable with the JSONL approach: records already saved are not re-fetched.
10. Add a data quality gate: if more than 10% of records are rejected, write nothing
    and report a failure.
11. Scrape `/shop.html` in the same run and save the products alongside, so the job
    combines an API source and a scraped source.

In [ ]:
# 1 and 2: session and api_get
import requests, time



In [ ]:
# 3: fetch_all



In [ ]:
# 4: flatten



In [ ]:
# 5: validate



In [ ]:
# 6 and 7: save and run



In [ ]:
# prove it: run twice, compare counts



---

## Common mistakes to watch for

### APIs

| Mistake | What you see | The fix |
|---|---|---|
| No `timeout` | The job hangs forever, silently | `timeout=10` on every call |
| Never checking the status | Empty results, no error | `raise_for_status()` |
| Calling `.json()` on an error page | `JSONDecodeError` | Check the status first |
| Building the query string by hand | Broken URLs when a value has a space | `params={...}` |
| `?pages=` instead of `?page=` | 200 OK every time, page 1 every time | Check the **distinct** count |
| Counting pages instead of following `next` | Off by one, or missing new pages | Follow the `next` link |
| No safety limit on a `while` loop | Infinite loop hammering someone's API | `max_pages`, and track seen URLs |
| Retrying a 401 or 400 | Wasted time, possible block | Retry 5xx and 429 only |
| Retrying with no delay | You make the outage worse | Exponential backoff |
| Ignoring `Retry-After` | You get blocked | Read the header, wait that long |
| A new connection per request | Slow over the internet | `requests.Session()` |
| API key in the notebook | Committed and leaked | Environment variables |
| Collecting everything in memory | Crash on page 400 loses all of it | Write to JSONL as you go |

### Scraping

| Mistake | What you see | The fix |
|---|---|---|
| `class=` instead of `class_` | `SyntaxError` | `class` is reserved in Python |
| Assuming a tag exists | `AttributeError: 'NoneType' has no attribute 'text'` | Check for `None`, use defaults |
| `.text` on a container | Everything glued together | Find the specific tag first |
| `BASE_URL + href` | Double slashes, broken links | `urljoin(base, href)` |
| No delay between requests | Blocked, or you take the site down | Sleep, honour `Crawl-delay` |
| No `User-Agent` | Blocked by default on many sites | Identify yourself honestly |
| Scraping a JavaScript page | BeautifulSoup finds nothing | Find the underlying API, or use Playwright |
| Re-fetching while developing | Slow, and rude | Cache the page locally |
| Never checking `robots.txt` | Avoidable trouble | Read it, and the terms of service |
| Writing a table parser by hand | Twenty lines that break | `pd.read_html` |

In [ ]:
# The two-second check that catches most pagination bugs
records = []
for page in range(1, 5):
    r = requests.get(f"{BASE_URL}/api/character", params={"page": page}, timeout=5)
    if r.status_code == 200:
        records.extend(r.json()["results"])

ids = [r["id"] for r in records]
print(f"records collected: {len(ids)}")
print(f"distinct ids:      {len(set(ids))}")

if len(ids) != len(set(ids)):
    print("DUPLICATES. Check your pagination parameter before trusting any of this.")
else:
    print("clean")

In [ ]:
# Clean up. Run this when you are finished.
import os

for name in ["characters.jsonl", "characters.csv", "character_episodes.csv",
             "rejected.csv", "run_summary.json", "products.csv"]:
    if os.path.isfile(name):
        os.remove(name)

try:
    _server.shutdown()
    print("practice API stopped")
except Exception:
    pass

print("cleaned up. re-run the setup cell at the top to start over")